# 2.2数据预处理

## 2.2.1. 读取数据集
举个例子，首先创建一个人工数据集，存储在CSV（逗号分隔符）文件 ../data/house_tiny.csv 中。以其他格式存储的数据也可以通过类似的方式进行处理。

In [2]:
import os

os.makedirs(os.path.join('.', 'data'), exist_ok=True)
data_file = os.path.join('.', 'data', 'house_tiny.csv')
with open(data_file, 'w') as f:
    f.write('NumRooms,Alley,Price\n')  # 列名
    f.write('NA,Pave,127500\n')  # 每行表示一个数据样本
    f.write('2,NA,106000\n')
    f.write('4,NA,178100\n')
    f.write('NA,NA,140000\n')

要从创建的CSV文件中加载原始数据集，我们导入pandas包并调用read_csv函数。

In [3]:
# 如果没有安装pandas，只需取消对以下行的注释来安装pandas
# !pip install pandas
import pandas as pd

data = pd.read_csv(data_file)
print(data)

   NumRooms Alley   Price
0       NaN  Pave  127500
1       2.0   NaN  106000
2       4.0   NaN  178100
3       NaN   NaN  140000


# 2.2.2. 处理缺失值
"NaN"代表缺失值。为了处理缺失值，带女性的方法包括插值法和删除法。插值法用一个替代值弥补缺失值，删除法则直接忽略缺失值。\
\
这里我们考虑插值法。\
\
通过位置索引iloc，我们将data分成inputs和outputs，其中前者为data的前两列，而后者为data的最后一列。对于input中缺少的数值，我们用同一列的均值替换"NaN"项。

In [8]:
inputs, outputs = data.iloc[:, 0:2], data.iloc[:, 2]
inputs = pd.get_dummies(inputs, dummy_na=True)
inputs = inputs.fillna(inputs.mean())
print(inputs)

   NumRooms  Alley_Pave  Alley_nan
0       3.0        True      False
1       2.0       False       True
2       4.0       False       True
3       3.0       False       True


对于inputs中的类别值或离散值，我们将"NaN"视为一个类别。由于"巷子类型"("Alley")列只接受两种类型的类别值"Pave"和"NaN"，pandas可以自动将此列转换为两列"Alley_Pave"和"Alley_nan"。巷子类型为"Pave"的行为会将"Alley_Pave"的值设置为1，"Alley_nan"的值设置为0。缺少巷子类型的行为会将"Alley_Pave"和"Alley_nan"分别设置为0和1。

In [9]:
inputs = pd.get_dummies(inputs, dummy_na=True)
print(inputs)

   NumRooms  Alley_Pave  Alley_nan
0       3.0        True      False
1       2.0       False       True
2       4.0       False       True
3       3.0       False       True


# 2.2.3. 转换为张量格式

In [10]:
import torch

X = torch.tensor(inputs.to_numpy(dtype=float))
y = torch.tensor(outputs.to_numpy(dtype=float))
X, y

(tensor([[3., 1., 0.],
         [2., 0., 1.],
         [4., 0., 1.],
         [3., 0., 1.]], dtype=torch.float64),
 tensor([127500., 106000., 178100., 140000.], dtype=torch.float64))

# 2.2.4. 小结
- pandas软件包是Python中常用的数据分析工具，pandas可以与张量兼容
- 用pandas处理缺失的数据时，可根据情况使用插值法和删除法。